<a href="https://colab.research.google.com/github/strangeworks/examples/blob/master/examples/optimization/quantagonia/quantagonia.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a>


# Quantagonia Optimization

This notebook demonstrates how to use Quantagonia's HybridSolver through the Strangeworks Optimization Service. Quantagonia provides hybrid quantum-classical optimization solutions that support QUBO, MPS, and LP problem formats.


# Installation


In [ ]:
%pip install -q -U pip numpy strangeworks-optimization


## Authentication

Your API token can be found in the [portal](https://portal.strangeworks.com).


### Google Colab

If running in Google Colab first set your API token as a secret environment variable in Colab. You can do this by clicking on the key icon on the left, then adding a key called `STRANGEWORKS_API_KEY` with your API token as the value.

Then, run the cell below to authenticate.


In [ ]:
import strangeworks as sw
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore', category=ImportWarning)

api_key = userdata.get('STRANGEWORKS_API_KEY')
sw.authenticate(api_key)


### Local

If running locally, you can save your API token in a dotenv file. Create a file called `.env` in the same directory as this notebook and add the following line to the file:

```
STRANGEWORKS_API_KEY=your_api_token
```

Then, run the cell below to authenticate.


In [ ]:
# %pip install -q python-dotenv
# import strangeworks as sw
# import os
# from dotenv import load_dotenv

# load_dotenv()
# api_key = os.getenv("STRANGEWORKS_API_KEY")
# sw.authenticate(api_key)


# StrangeworksOptimizer

We implement our own Optimization class called `StrangeworksOptimizer`.
`StrangeworksOptimizer` provides the full functionality of the Strangeworks Optimization Service.


In [ ]:
from strangeworks_optimization import StrangeworksOptimizer
so = StrangeworksOptimizer()


## Models

Quantagonia accepts the following models:

- `BinaryQuadraticModel` - [`dimod`](https://docs.ocean.dwavesys.com/en/stable/docs_dimod/) provides a shared API for samplers and we default to using `BinaryQuadraticModel` to store and represent QUBO problems. Use with `quantagonia.qubo` solver.
- `MPSFile` - Mathematical Programming System format, a standard format for linear and mixed-integer programming problems. Use with `quantagonia.mps` solver. **Note:** Quantagonia does NOT support MIQCP (Mixed-Integer Quadratically Constrained Programming) problems. Only linear and mixed-integer linear programming problems are supported in MPS format.
- `LpFile` - Linear Programming format. Use with `quantagonia.lp` solver.

Here is a simple QUBO problem defined by linear and quadratic terms. The linear terms are a dictionary where the keys are the variables and the values are the linear coefficients. The quadratic terms are a dictionary where the keys are the pairs of variables and the values are the quadratic coefficients. The BinaryQuadraticModel class is used to create the QUBO problem.


In [ ]:
from dimod import BinaryQuadraticModel

linear = {1: -2, 2: -2, 3: -3, 4: -3, 5: -2}
quadratic = {(1, 2): 2, (1, 3): 2, (2, 4): 2, (3, 4): 2, (3, 5): 2, (4, 5): 2}
model = BinaryQuadraticModel(linear, quadratic, "BINARY")


## Solvers

The solver name usually follows the convention of `provider-id.backend-name`.

Quantagonia has the following solvers:
- `quantagonia.qubo` - For Quadratic Unconstrained Binary Optimization (QUBO) problems
- `quantagonia.mps` - For Mixed Integer Linear Programming (MILP) problems using MPS format (note: MIQCP is not supported)
- `quantagonia.lp` - For Linear Programming (LP) format problems


## Options

Quantagonia provides parameter control through `QuantagoniaParameterModel`. Some commonly used parameters include:

**Optimization Direction:**
- `sense`: Determines the optimization direction (`MINIMIZE` or `MAXIMIZE`)

**Termination Parameters:**
- `timelimit`: Maximum time (in seconds) the solver should run
- `relative_gap`: Stopping criterion based on the relative gap between the best-found solution and the bound
- `absolute_gap`: Absolute gap tolerance

**QUBO-Specific Parameters:**
- `heuristics_only`: If set to `true`, the solver will only use heuristic methods (only for QUBO problems)


In [ ]:
solver = "quantagonia.qubo"

from strangeworks_optimization_models.parameter_models import QuantagoniaParameterModel

options = QuantagoniaParameterModel(sense="MINIMIZE")


## Running a job


In [ ]:
tags = ["foo", "bar"]
so = StrangeworksOptimizer(model=model, solver=solver, options=options, tags=tags)
sw_job = so.run()
print(f"Job slug: {sw_job.slug}")


## Status


In [ ]:
print(f"Job status: {so.status(sw_job.slug)}")


## Results


In [ ]:
results = so.results(sw_job.slug)
print(f"Best solution:\n{results.solution.first}")


## Additional Examples

### MPS Format

Quantagonia also supports MPS (Mathematical Programming System) format, which is a standard format for linear and mixed-integer programming problems:

**Important Note:** Quantagonia does NOT support MIQCP (Mixed-Integer Quadratically Constrained Programming) problems. Only linear and mixed-integer linear programming (MILP) problems are supported in MPS format. If you need to solve MIQCP problems, consider using Gurobi, which has full MIQCP support (see the [Gurobi MIQCP documentation](https://www.gurobi.com/faqs/gurobi-miqcp/)).

```python
from strangeworks_optimization import StrangeworksOptimizer
from strangeworks_optimization_models.problem_models import MPSFile
from strangeworks_optimization_models.parameter_models import QuantagoniaParameterModel

# Read an MPS file
# model = MPSFile.read_file("example.mps")

# solver = "quantagonia.mps"
# options = QuantagoniaParameterModel(sense="MINIMIZE")

# so = StrangeworksOptimizer(model=model, solver=solver, options=options)
# sw_job = so.run()
# results = so.results(sw_job.slug)
```

### LP Format

Quantagonia also supports LP (Linear Programming) format:

```python
from strangeworks_optimization import StrangeworksOptimizer
from strangeworks_optimization_models.problem_models import LpFile
from strangeworks_optimization_models.parameter_models import QuantagoniaParameterModel

# Read an LP file
# model = LpFile.read_file("example.lp")
# options = QuantagoniaParameterModel(sense="MINIMIZE")

# so = StrangeworksOptimizer(model=model, solver="quantagonia.lp", options=options)
# sw_job = so.run()
# results = so.results(sw_job.slug)
```
